In [1]:
from datasets import load_dataset
import os

# Load dataset
ds = load_dataset("Renicames/turkish-law-chatbot")

print(ds)
print(ds["train"].column_names)

# Split original train into train + validation
split_data = ds["train"].train_test_split(
    test_size=0.1,
    seed=42,
    shuffle=True
)

train_ds = split_data["train"]
val_ds = split_data["test"]

# Use original test split if it exists
test_ds = ds["test"] if "test" in ds else None

# Create local folder
save_dir = "data/renicames_turkish_law_chatbot"
os.makedirs(save_dir, exist_ok=True)

# Save as JSONL
train_ds.to_json(f"{save_dir}/train.jsonl", force_ascii=False)
val_ds.to_json(f"{save_dir}/validation.jsonl", force_ascii=False)

if test_ds is not None:
    test_ds.to_json(f"{save_dir}/test.jsonl", force_ascii=False)

print("Saved files to:", save_dir)
print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))
if test_ds is not None:
    print("Test size:", len(test_ds))

DatasetDict({
    train: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 13354
    })
    test: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 1500
    })
})
['Soru', 'Cevap']


Creating json from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Saved files to: data/renicames_turkish_law_chatbot
Train size: 12018
Validation size: 1336
Test size: 1500


In [2]:
from datasets import load_dataset
import os
import pandas as pd

ds = load_dataset("Renicames/turkish-law-chatbot")
print(ds)
print(ds["train"][0])  # See the column names and structure

DatasetDict({
    train: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 13354
    })
    test: Dataset({
        features: ['Soru', 'Cevap'],
        num_rows: 1500
    })
})
{'Soru': "Anayasa madde 1'e göre, türkiye'nin devlet şekli nedir", 'Cevap': "Anayasa madde 1'e göre, türkiye'nin devlet şekli cumhuriyettir. bu madde, türkiye'nin yönetim biçiminin halkın egemenliğine dayandığını ve bu yönetim biçiminin cumhuriyet olduğunu belirler. cumhuriyet, halkın kendi kendini yönetme biçimi olarak kabul edilir ve türkiye cumhuriyeti'nin temel yönetim ilkesi olarak anayasal güvence altına alınmıştır."}


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

# ── Load model ──────────────────────────────────────────────────────────────
model_Qwen = "Qwen/Qwen2.5-1.5B-Instruct"
Qwen_tokenizer = AutoTokenizer.from_pretrained(model_Qwen)
Qwen_model = AutoModelForCausalLM.from_pretrained(
    model_Qwen,
    torch_dtype="auto",
    device_map="auto"
)

# ── Load dataset ─────────────────────────────────────────────────────────────
ds = load_dataset("Renicames/turkish-law-chatbot")
split = ds["train"]  # or "test" if available

# ── Inference function ────────────────────────────────────────────────────────
def ask_Qwen(question: str, system_prompt: str = "Sen Türk hukuku konusunda uzman bir asistansın.") -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = Qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = Qwen_tokenizer([text], return_tensors="pt").to(Qwen_model.device)
    with torch.no_grad():
        generated_ids = Qwen_model.generate(**inputs, max_new_tokens=512)
    generated_ids = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)
    ]
    return Qwen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# ── Evaluate on N samples ─────────────────────────────────────────────────────
N = 40  # start small to test quickly
Qwen_results = []
question_list = split.select(range(N))

for i, sample in enumerate(question_list):
    question = sample["Soru"]   # ← adjust column name if needed
    reference = sample["Cevap"]       # ← adjust column name if needed

    prediction = ask_Qwen(question)

    Qwen_results.append({
        "id": i,
        "model": "Qwen2.5-1.5B-Instruct",
        "question": question,
        "reference": reference,
        "prediction": prediction
    })
    
    qwen_quest_and_preds = f"Q: {question}\nPred: {prediction}\nRef: {reference}\n"
    
    print(f"[{i+1}/{N}] Q: {question[:80]}...")
    print(f"       Pred: {prediction[:120]}...\n")

# ── (Optional) Compute ROUGE scores ──────────────────────────────────────────
# pip install rouge-score
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
scores = [scorer.score(r["reference"], r["prediction"]) for r in Qwen_results]

avg_r1 = sum(s["rouge1"].fmeasure for s in scores) / len(scores)
avg_rL = sum(s["rougeL"].fmeasure for s in scores) / len(scores)

qwen_results = {
    "results": Qwen_results,
    "scores": scores,
    "avg_rouge1": avg_r1,
    "avg_rougeL": avg_rL,
}

row_df = pd.DataFrame(qwen_results)
file_exists = os.path.isfile("qwen_results.csv")
row_df.to_csv("qwen_results.csv", mode='w', index=False, encoding='utf-8-sig')

print(f"\nAvg ROUGE-1: {avg_r1:.4f}")
print(f"Avg ROUGE-L: {avg_rL:.4f}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[1/40] Q: Anayasa madde 1'e göre, türkiye'nin devlet şekli nedir...
       Pred: Türkiye'nin devlet şekli, Anayasada belirlenmiş "Devlet Hakkı" (Artık Anayasa Madde 1) tarafından tanımlanmıştır. Bu anl...

[2/40] Q: Anayasa madde 1'de belirtilen cumhuriyetin tanımı nedir...
       Pred: Anayasa Madde 1, Cumhuriyet'in tanımını ve temel kurumlarını belirtmektedir. Bu anayasa, 1982 yılında Türk Cumhuriyeti'n...

[3/40] Q: Anayasa madde 1, cumhuriyetin ilan edilmesini neden önemli görmüştür...
       Pred: Anayasa madde 1'nin cumhuriyetin ilan edilmesi, Türkiye'de önemli bir hukuki öneme sahiptir çünkü:

1. Yeni bir anayasay...

[4/40] Q: Anayasa madde 1, cumhuriyetin hangi tarihte ilan edildiğini belirtir mi...
       Pred: Türkiye'nin Anayasası Madde 1'ü, Cumhuriyet'in kurulmasından 5 yıl önce (1923) ilan edilmiştir. Bu madde, Türkiye Cumhur...

[5/40] Q: Anayasa madde 1'e göre, cumhuriyetin temel özellikleri nelerdir...
       Pred: Anayasa Madde 1'nin temel özellikler şunlardır:

1. Cu

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

# ── Load model ──────────────────────────────────────────────────────────────
model_Qwen = "Qwen/Qwen2.5-3B-Instruct"
Qwen_tokenizer = AutoTokenizer.from_pretrained(model_Qwen)
Qwen_model = AutoModelForCausalLM.from_pretrained(
    model_Qwen,
    torch_dtype="auto",
    device_map="auto"
)

# ── Load dataset ─────────────────────────────────────────────────────────────
ds = load_dataset("Renicames/turkish-law-chatbot")
split = ds["train"]  # or "test" if available

# ── Inference function ────────────────────────────────────────────────────────
def ask_Qwen(question: str, system_prompt: str = "Sen Türk hukuku konusunda uzman bir asistansın.") -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = Qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = Qwen_tokenizer([text], return_tensors="pt").to(Qwen_model.device)
    with torch.no_grad():
        generated_ids = Qwen_model.generate(**inputs, max_new_tokens=512)
    generated_ids = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)
    ]
    return Qwen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# ── Evaluate on N samples ─────────────────────────────────────────────────────
N = 40  # start small to test quickly
Qwen_results = []
question_list = split.select(range(N))

for i, sample in enumerate(question_list):
    question = sample["Soru"]   # ← adjust column name if needed
    reference = sample["Cevap"]       # ← adjust column name if needed

    prediction = ask_Qwen(question)

    Qwen_results.append({
        "id": i,
        "model": "Qwen2.5-3",
        "question": question,
        "reference": reference,
        "prediction": prediction
    })
    
    qwen_quest_and_preds = f"Q: {question}\nPred: {prediction}\nRef: {reference}\n"
    
    print(f"[{i+1}/{N}] Q: {question[:80]}...")
    print(f"       Pred: {prediction[:120]}...\n")

# ── (Optional) Compute ROUGE scores ──────────────────────────────────────────
# pip install rouge-score
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
scores = [scorer.score(r["reference"], r["prediction"]) for r in Qwen_results]

avg_r1 = sum(s["rouge1"].fmeasure for s in scores) / len(scores)
avg_rL = sum(s["rougeL"].fmeasure for s in scores) / len(scores)

qwen_results = {
    "results": Qwen_results,
    "scores": scores,
    "avg_rouge1": avg_r1,
    "avg_rougeL": avg_rL,
}

row_df = pd.DataFrame(qwen_results)
file_exists = os.path.isfile("qwen2.5-3_results.csv")
row_df.to_csv("qwen2.5-3_results.csv", mode='w', index=False, encoding='utf-8-sig')

print(f"\nAvg ROUGE-1: {avg_r1:.4f}")
print(f"Avg ROUGE-L: {avg_rL:.4f}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
import numpy as np
import re
import ast
from rouge_score import rouge_scorer
from bert_score import score as bert_score



#file_path = "qwen_results.csv"
file_path = "qwen2.5-3_results.csv"

df = pd.read_csv(file_path)

print("Original columns:")
print(df.columns.tolist())
print(df.head())




if "results" in df.columns:
    print("\nNested 'results' column found. Flattening...")

    # Convert string dictionaries to real Python dictionaries
    df["results"] = df["results"].apply(ast.literal_eval)

    # Convert dictionaries into normal dataframe columns
    df = pd.json_normalize(df["results"])

else:
    print("\nNo nested 'results' column found. Using CSV as-is.")

print("\nColumns after flattening:")
print(df.columns.tolist())
print(df.head())




possible_question_cols = [
    "question", "Question", "Soru", "soru", "instruction", "input"
]

possible_reference_cols = [
    "reference", "Reference", "Cevap", "cevap", "answer", "Answer", "output"
]

possible_prediction_cols = [
    "prediction", "Prediction", "model_answer", "response",
    "generated_answer", "pred", "Prediction"
]


def find_col(possible_cols, actual_cols):
    for col in possible_cols:
        if col in actual_cols:
            return col
    return None


QUESTION_COL = find_col(possible_question_cols, df.columns)
REFERENCE_COL = find_col(possible_reference_cols, df.columns)
PREDICTION_COL = find_col(possible_prediction_cols, df.columns)

print("\nDetected columns:")
print("QUESTION_COL:", QUESTION_COL)
print("REFERENCE_COL:", REFERENCE_COL)
print("PREDICTION_COL:", PREDICTION_COL)

if REFERENCE_COL is None or PREDICTION_COL is None:
    raise ValueError(
        f"Could not detect reference/prediction columns. "
        f"Available columns are: {df.columns.tolist()}"
    )



df = df.dropna(subset=[REFERENCE_COL, PREDICTION_COL]).reset_index(drop=True)

print("\nRows after dropping empty reference/prediction:", len(df))




def normalize_text(text):
    """
    Simple Turkish-friendly normalization.
    Keeps Turkish characters but removes punctuation and extra spaces.
    """
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\sçğıöşüâîû]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()



rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for _, row in df.iterrows():
    reference = str(row[REFERENCE_COL])
    prediction = str(row[PREDICTION_COL])

    scores = rouge.score(reference, prediction)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["rouge1"] = rouge1_scores
df["rouge2"] = rouge2_scores
df["rougeL"] = rougeL_scores

print("\nROUGE results:")
print("Avg ROUGE-1:", np.mean(rouge1_scores))
print("Avg ROUGE-2:", np.mean(rouge2_scores))
print("Avg ROUGE-L:", np.mean(rougeL_scores))




exact_matches = []

for _, row in df.iterrows():
    ref_norm = normalize_text(row[REFERENCE_COL])
    pred_norm = normalize_text(row[PREDICTION_COL])

    exact_matches.append(1 if ref_norm == pred_norm else 0)

df["exact_match"] = exact_matches

print("\nExact Match:", np.mean(exact_matches))



def token_f1(reference, prediction):
    ref_tokens = normalize_text(reference).split()
    pred_tokens = normalize_text(prediction).split()

    if len(ref_tokens) == 0 and len(pred_tokens) == 0:
        return 1.0

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0

    common_tokens = set(ref_tokens) & set(pred_tokens)

    common_count = sum(
        min(ref_tokens.count(tok), pred_tokens.count(tok))
        for tok in common_tokens
    )

    if common_count == 0:
        return 0.0

    precision = common_count / len(pred_tokens)
    recall = common_count / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


df["token_f1"] = df.apply(
    lambda row: token_f1(row[REFERENCE_COL], row[PREDICTION_COL]),
    axis=1
)

print("Avg Token F1:", df["token_f1"].mean())




df["prediction_word_count"] = df[PREDICTION_COL].astype(str).apply(
    lambda x: len(x.split())
)

df["reference_word_count"] = df[REFERENCE_COL].astype(str).apply(
    lambda x: len(x.split())
)

print("\nLength statistics:")
print("Average prediction length:", df["prediction_word_count"].mean())
print("Average reference length:", df["reference_word_count"].mean())
print("Max prediction length:", df["prediction_word_count"].max())
print("Min prediction length:", df["prediction_word_count"].min())



suspicious_patterns = [
    "emin değilim",
    "genellikle",
    "olabilir",
    "muhtemelen",
    "varsayım",
    "belirtilmemiştir",
    "bilinmemektedir",
    "kaynaklara göre",
    "şu şekilde sıralanabilir",
    "örneğin",
    "bazı durumlarda",
    "genel olarak"
]


def flag_suspicious_answer(text):
    text = str(text).lower()
    return int(any(pattern in text for pattern in suspicious_patterns))


df["suspicious_flag"] = df[PREDICTION_COL].apply(flag_suspicious_answer)

print("\nSuspicious answer check:")
print("Suspicious answer ratio:", df["suspicious_flag"].mean())
print("Suspicious answer count:", df["suspicious_flag"].sum())


if "qwen2.5-3" in file_path.lower():
    evaluated_path = "qwen2.5-3_results_evaluated.csv"
    summary_path = "qwen2.5-3_results_summary.csv"
elif "llama" in file_path.lower():
    evaluated_path = "llama_results_evaluated.csv"
    summary_path = "llama_results_summary.csv"
elif "qwen" in file_path.lower():
    evaluated_path = "qwen_results_evaluated.csv"
    summary_path = "qwen_results_summary.csv"
else:
    evaluated_path = file_path.replace(".csv", "_evaluated.csv")
    summary_path = file_path.replace(".csv", "_summary.csv")

df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")

print("\nSaved evaluated file to:", evaluated_path)


summary = {
    "file": file_path,
    "num_samples": len(df),
    "rouge1": df["rouge1"].mean(),
    "rouge2": df["rouge2"].mean(),
    "rougeL": df["rougeL"].mean(),
    "exact_match": df["exact_match"].mean(),
    "token_f1": df["token_f1"].mean(),
    "avg_prediction_words": df["prediction_word_count"].mean(),
    "avg_reference_words": df["reference_word_count"].mean(),
    "suspicious_ratio": df["suspicious_flag"].mean(),
    "suspicious_count": df["suspicious_flag"].sum()
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved summary file to:", summary_path)

print("\nSummary:")
print(summary_df)

Original columns:
['results', 'scores', 'avg_rouge1', 'avg_rougeL']
                                             results  \
0  {'id': 0, 'model': 'Qwen2.5-1.5B-Instruct', 'q...   
1  {'id': 1, 'model': 'Qwen2.5-1.5B-Instruct', 'q...   
2  {'id': 2, 'model': 'Qwen2.5-1.5B-Instruct', 'q...   
3  {'id': 3, 'model': 'Qwen2.5-1.5B-Instruct', 'q...   
4  {'id': 4, 'model': 'Qwen2.5-1.5B-Instruct', 'q...   

                                              scores  avg_rouge1  avg_rougeL  
0  {'rouge1': Score(precision=0.22602739726027396...     0.26727    0.164773  
1  {'rouge1': Score(precision=0.16585365853658537...     0.26727    0.164773  
2  {'rouge1': Score(precision=0.11594202898550725...     0.26727    0.164773  
3  {'rouge1': Score(precision=0.5862068965517241,...     0.26727    0.164773  
4  {'rouge1': Score(precision=0.15196078431372548...     0.26727    0.164773  

Nested 'results' column found. Flattening...

Columns after flattening:
['id', 'model', 'question', 'reference', 'predic

In [6]:
import pandas as pd

qwen_summary = pd.read_csv("qwen_results_summary.csv")
llama_summary = pd.read_csv("llama_results_summary.csv")

comparison = pd.concat([qwen_summary, llama_summary], ignore_index=True)

print(comparison.columns.tolist())
comparison

['file', 'num_samples', 'rouge1', 'rouge2', 'rougeL', 'exact_match', 'token_f1', 'avg_prediction_words', 'avg_reference_words', 'suspicious_ratio', 'suspicious_count']


,file,num_samples,rouge1,rouge2,rougeL,exact_match,token_f1,avg_prediction_words,avg_reference_words,suspicious_ratio,suspicious_count
0,qwen3_results.csv,40,0.062201,0.022859,0.044660,0.0,0.033044,396.625,54.625,0.025,1
1,llama_results.csv,40,0.169587,0.059738,0.128739,0.0,0.136095,99.350,54.625,0.000,0
